# 06 - Discharge Destination and Long-Stay Burden in MIMIC-IV

Author: Saige Mukherjee

Contact: mukherjeesaige@gmail.com //
https://www.linkedin.com/in/saige-mukherjee-0aba68281/

The analysis runs inside the Jupyter notebook.

If you want to run the notebook and generate the report:
- obtained credentialed access to MIMIC-IV via PhysioNet and BigQuery,
- create a GCP project and star the MIMIC-IV dataset ,
- enter the GCP project ID below and execute the program.

In [ ]:
PROJECT_ID = "" # enter your GCP project ID into the string

## Purpose

This notebook tests whether admissions requiring **coordinated downstream care** are associated with a disproportionate share of prolonged acute-hospital bed use at Beth Israel Deaconess Medical Center (BIDMC).

It links:

- `admissions`: hospital admission/discharge times and eventual `discharge_location`
- `transfers`: physical care-unit movements
- `services`: the clinical service responsible near discharge
- `icustays`: whether the hospitalization included ICU care

The analysis asks:

1. Which discharge destinations have the longest hospital stays?
2. Which destinations account for the most excess bed-hours above 48, 72, 168, and 336 hours?
3. Are next-care destinations overrepresented in the long-stay burden relative to their admission volume?
4. Do those patterns also appear in the **final physical care unit**?
5. Are they repeated within the same final clinical service and ICU-status stratum?

## Inference boundary

A result showing that skilled nursing, rehabilitation, long-term acute care, psychiatric facilities, home health, or other coordinated destinations account for disproportionate excess bed-hours would be **consistent with a downstream-care bottleneck**.

It would not prove that every additional hour was spent waiting for placement. MIMIC-IV does not provide a reliable “medically ready for discharge” timestamp, referral time, authorization time, facility acceptance time, or downstream bed-availability time.

## References

- Johnson, A. E. W., et al. *MIMIC-IV, a freely accessible electronic health record dataset.* Scientific Data 10, 1 (2023). DOI: `10.1038/s41597-022-01899-x`
- Johnson, A., et al. *MIMIC-IV (version 3.1).* PhysioNet (2024). DOI: `10.13026/kpb9-mt58`

## Privacy and publication rules

- No patient-level rows are returned to Python or displayed.
- Every displayed result is aggregated.
- Any grouped result with `0 < n < 11` is removed in SQL.
- A count of `0` is acceptable.
- No CSV files or derived datasets are written.
- `subject_id` and `hadm_id` are used only inside BigQuery to perform joins.
- Missing `discharge_location` is retained as **UNKNOWN**, not assumed to be home.

In [ ]:
%pip -q install google-cloud-bigquery google-cloud-bigquery-storage db-dtypes pandas numpy matplotlib

In [ ]:
from __future__ import annotations

from typing import Iterable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from google.cloud import bigquery

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## 1. Authenticate and configure BigQuery

Set `PROJECT_ID` to a Google Cloud project with BigQuery billing enabled and linked to your PhysioNet access.

In [ ]:
HOSP_DATASET = "physionet-data.mimiciv_3_1_hosp"
ICU_DATASET = "physionet-data.mimiciv_3_1_icu"

MIN_CELL_SIZE = 11
THRESHOLDS_HOURS = (48, 72, 168, 336)

client = bigquery.Client(project=PROJECT_ID)

In [ ]:
def run_query(sql: str, label: str = "Query") -> pd.DataFrame:
    """Run an aggregate BigQuery query and return a DataFrame."""
    print(f"Running: {label}")
    job_config = bigquery.QueryJobConfig(use_query_cache=True)
    df = client.query(sql, job_config=job_config).to_dataframe(
        create_bqstorage_client=True
    )
    print(f"Returned {len(df):,} aggregate rows.")
    return df


def assert_no_small_cells(
    df: pd.DataFrame,
    count_columns: Iterable[str] = ("n_admissions",),
    minimum: int = MIN_CELL_SIZE,
) -> None:
    """Raise an error if a displayed aggregate contains a count from 1 to 10."""
    for column in count_columns:
        if column not in df.columns:
            continue
        values = pd.to_numeric(df[column], errors="coerce")
        bad = values[(values > 0) & (values < minimum)]
        if not bad.empty:
            raise ValueError(
                f"Small-cell suppression failure in {column}: "
                f"found a count between 1 and {minimum - 1}."
            )


def display_aggregate(
    df: pd.DataFrame,
    count_columns: Iterable[str] = ("n_admissions",),
    rows: int | None = None,
) -> None:
    """Check small cells, then display an aggregate table."""
    assert_no_small_cells(df, count_columns=count_columns)
    display(df if rows is None else df.head(rows))


def horizontal_bar(
    df: pd.DataFrame,
    category: str,
    value: str,
    title: str,
    xlabel: str,
    top_n: int | None = None,
) -> None:
    """Create a readable horizontal bar chart."""
    plot_df = df[[category, value]].dropna().copy()
    if top_n is not None:
        plot_df = plot_df.nlargest(top_n, value)
    plot_df = plot_df.sort_values(value, ascending=True)

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(plot_df))))
    ax.barh(plot_df[category].astype(str), plot_df[value])
    ax.set_title(title)
    ax.set_xlabel(xlabel)
    ax.set_ylabel("")
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()

## 2. Admission-level logic kept inside BigQuery

### Measures

**Hospital length of stay**

\[
	ext{hospital LOS} = 	ext{dischtime} - 	ext{admittime}
\]

**Final-unit dwell**

The last non-discharge physical transfer segment is selected for each hospitalization:

\[
	ext{final-unit dwell} =
\min(	ext{transfer outtime}, 	ext{hospital dischtime}) -
\max(	ext{transfer intime}, 	ext{hospital admittime})
\]

**Excess bed-hours above threshold \(T\)**

\[
	ext{excess hours}_T = \max(	ext{duration} - T, 0)
\]

Excess hours are a burden measure, not a claim that every hour above the threshold was avoidable.

In [ ]:
def make_base_ctes() -> str:
    return f"""
WITH admissions_base AS (
    SELECT
        a.subject_id,
        a.hadm_id,
        a.admittime,
        a.dischtime,
        a.admission_type,
        a.hospital_expire_flag,
        COALESCE(a.discharge_location, 'UNKNOWN') AS discharge_location,
        SAFE_DIVIDE(
            TIMESTAMP_DIFF(a.dischtime, a.admittime, MINUTE),
            60.0
        ) AS hospital_los_hours
    FROM `{HOSP_DATASET}.admissions` AS a
    WHERE a.hadm_id IS NOT NULL
      AND a.admittime IS NOT NULL
      AND a.dischtime IS NOT NULL
      AND a.dischtime > a.admittime
),

ranked_physical_segments AS (
    SELECT
        t.hadm_id,
        t.transfer_id,
        t.eventtype,
        t.careunit,
        t.intime,
        t.outtime,
        ROW_NUMBER() OVER (
            PARTITION BY t.hadm_id
            ORDER BY
                COALESCE(t.outtime, a.dischtime) DESC,
                t.intime DESC,
                t.transfer_id DESC
        ) AS segment_rank
    FROM `{HOSP_DATASET}.transfers` AS t
    INNER JOIN admissions_base AS a
        USING (hadm_id)
    WHERE t.careunit IS NOT NULL
      AND t.intime IS NOT NULL
      AND LOWER(COALESCE(t.eventtype, '')) != 'discharge'
      AND t.intime < a.dischtime
      AND COALESCE(t.outtime, a.dischtime) > a.admittime
),

final_segment AS (
    SELECT
        hadm_id,
        careunit AS final_careunit,
        eventtype AS final_eventtype,
        intime AS final_unit_intime,
        outtime AS final_unit_outtime
    FROM ranked_physical_segments
    WHERE segment_rank = 1
),

ranked_services AS (
    SELECT
        s.hadm_id,
        s.curr_service,
        s.transfertime,
        ROW_NUMBER() OVER (
            PARTITION BY s.hadm_id
            ORDER BY s.transfertime DESC, s.curr_service
        ) AS service_rank
    FROM `{HOSP_DATASET}.services` AS s
    INNER JOIN admissions_base AS a
        USING (hadm_id)
    WHERE s.transfertime IS NOT NULL
      AND s.transfertime <= a.dischtime
),

final_service AS (
    SELECT
        hadm_id,
        curr_service AS final_service
    FROM ranked_services
    WHERE service_rank = 1
),

icu_flag AS (
    SELECT DISTINCT
        hadm_id,
        1 AS had_icu
    FROM `{ICU_DATASET}.icustays`
    WHERE hadm_id IS NOT NULL
),

analysis_base AS (
    SELECT
        a.hadm_id,
        a.admission_type,
        a.hospital_expire_flag,
        a.discharge_location,
        a.hospital_los_hours,
        fs.final_careunit,
        svc.final_service,
        COALESCE(i.had_icu, 0) AS had_icu,

        CASE
            WHEN fs.final_unit_intime IS NULL THEN NULL
            ELSE SAFE_DIVIDE(
                TIMESTAMP_DIFF(
                    LEAST(
                        COALESCE(fs.final_unit_outtime, a.dischtime),
                        a.dischtime
                    ),
                    GREATEST(fs.final_unit_intime, a.admittime),
                    MINUTE
                ),
                60.0
            )
        END AS final_unit_hours,

        CASE
            WHEN fs.final_unit_outtime IS NULL THEN NULL
            ELSE SAFE_DIVIDE(
                TIMESTAMP_DIFF(a.dischtime, fs.final_unit_outtime, MINUTE),
                60.0
            )
        END AS terminal_gap_hours,

        CASE
            WHEN a.discharge_location = 'HOME'
                THEN 'HOME'
            WHEN a.discharge_location = 'HOME HEALTH CARE'
                THEN 'HOME_WITH_SERVICES'
            WHEN a.discharge_location IN (
                'SKILLED NURSING FACILITY',
                'REHAB',
                'CHRONIC/LONG TERM ACUTE CARE',
                'ASSISTED LIVING',
                'HEALTHCARE FACILITY',
                'OTHER FACILITY'
            )
                THEN 'POST_ACUTE_INSTITUTIONAL'
            WHEN a.discharge_location = 'PSYCH FACILITY'
                THEN 'PSYCHIATRIC_FACILITY'
            WHEN a.discharge_location = 'ACUTE HOSPITAL'
                THEN 'ACUTE_TRANSFER'
            WHEN a.discharge_location = 'HOSPICE'
                THEN 'HOSPICE'
            WHEN a.discharge_location = 'DIED'
                THEN 'IN_HOSPITAL_DEATH'
            WHEN a.discharge_location = 'AGAINST ADVICE'
                THEN 'AGAINST_ADVICE'
            WHEN a.discharge_location = 'UNKNOWN'
                THEN 'UNKNOWN'
            ELSE 'OTHER_UNKNOWN'
        END AS pathway_group,

        CASE
            WHEN a.discharge_location = 'HOME'
                THEN 'HOME_BASELINE'
            WHEN a.discharge_location IN (
                'HOME HEALTH CARE',
                'SKILLED NURSING FACILITY',
                'REHAB',
                'CHRONIC/LONG TERM ACUTE CARE',
                'PSYCH FACILITY',
                'ACUTE HOSPITAL',
                'OTHER FACILITY',
                'ASSISTED LIVING',
                'HEALTHCARE FACILITY',
                'HOSPICE'
            )
                THEN 'COORDINATED_NEXT_CARE'
            WHEN a.discharge_location IN ('DIED', 'AGAINST ADVICE')
                THEN 'NON_COMPARABLE_OUTCOME'
            WHEN a.discharge_location = 'UNKNOWN'
                THEN 'UNKNOWN'
            ELSE 'OTHER_UNKNOWN'
        END AS comparison_group

    FROM admissions_base AS a
    LEFT JOIN final_segment AS fs
        USING (hadm_id)
    LEFT JOIN final_service AS svc
        USING (hadm_id)
    LEFT JOIN icu_flag AS i
        USING (hadm_id)
)
"""

## 3. Coverage and data-quality checks

This checks:

- valid positive-duration admissions
- unknown discharge destination
- final-care-unit and final-service coverage
- alignment between final transfer `outtime` and hospital `dischtime`
- extreme duration percentiles that could dominate excess-hour totals

In [ ]:
quality_sql = make_base_ctes() + """
SELECT
    COUNT(*) AS n_valid_admissions,
    COUNTIF(discharge_location = 'UNKNOWN') AS n_unknown_destination,
    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(discharge_location = 'UNKNOWN'),
        COUNT(*)
    ), 2) AS pct_unknown_destination,

    COUNTIF(final_careunit IS NOT NULL) AS n_with_final_careunit,
    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(final_careunit IS NOT NULL),
        COUNT(*)
    ), 2) AS pct_with_final_careunit,

    COUNTIF(final_service IS NOT NULL) AS n_with_final_service,
    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(final_service IS NOT NULL),
        COUNT(*)
    ), 2) AS pct_with_final_service,

    COUNTIF(
        terminal_gap_hours IS NOT NULL
        AND ABS(terminal_gap_hours) <= 1
    ) AS n_terminal_gap_within_1h,

    ROUND(100 * SAFE_DIVIDE(
        COUNTIF(
            terminal_gap_hours IS NOT NULL
            AND ABS(terminal_gap_hours) <= 1
        ),
        COUNTIF(terminal_gap_hours IS NOT NULL)
    ), 2) AS pct_terminal_gap_within_1h,

    ROUND(APPROX_QUANTILES(hospital_los_hours, 1000)[OFFSET(500)], 2)
        AS hospital_los_p50_hours,
    ROUND(APPROX_QUANTILES(hospital_los_hours, 1000)[OFFSET(950)], 2)
        AS hospital_los_p95_hours,
    ROUND(APPROX_QUANTILES(hospital_los_hours, 1000)[OFFSET(990)], 2)
        AS hospital_los_p99_hours,
    ROUND(APPROX_QUANTILES(hospital_los_hours, 1000)[OFFSET(999)], 2)
        AS hospital_los_p999_hours,
    ROUND(MAX(hospital_los_hours), 2) AS hospital_los_max_hours,

    ROUND(APPROX_QUANTILES(final_unit_hours, 1000)[OFFSET(500)], 2)
        AS final_unit_p50_hours,
    ROUND(APPROX_QUANTILES(final_unit_hours, 1000)[OFFSET(950)], 2)
        AS final_unit_p95_hours,
    ROUND(APPROX_QUANTILES(final_unit_hours, 1000)[OFFSET(990)], 2)
        AS final_unit_p99_hours,
    ROUND(APPROX_QUANTILES(final_unit_hours, 1000)[OFFSET(999)], 2)
        AS final_unit_p999_hours,
    ROUND(MAX(final_unit_hours), 2) AS final_unit_max_hours
FROM analysis_base
"""

quality_df = run_query(quality_sql, "Coverage and quality checks")
display_aggregate(
    quality_df,
    count_columns=(
        "n_valid_admissions",
        "n_unknown_destination",
        "n_with_final_careunit",
        "n_with_final_service",
        "n_terminal_gap_within_1h",
    ),
)

## 4. Reproduce the raw discharge-location distribution

In [ ]:
destination_counts_sql = f"""
WITH counts AS (
    SELECT
        COALESCE(discharge_location, 'UNKNOWN') AS discharge_location,
        COUNT(*) AS n_admissions
    FROM `{HOSP_DATASET}.admissions`
    GROUP BY discharge_location
),
totals AS (
    SELECT SUM(n_admissions) AS total_admissions
    FROM counts
)
SELECT
    c.discharge_location,
    c.n_admissions,
    ROUND(
        100 * SAFE_DIVIDE(c.n_admissions, t.total_admissions),
        2
    ) AS pct_admissions
FROM counts AS c
CROSS JOIN totals AS t
WHERE c.n_admissions = 0 OR c.n_admissions >= {MIN_CELL_SIZE}
ORDER BY c.n_admissions DESC
"""

destination_counts_df = run_query(
    destination_counts_sql,
    "Raw discharge-location distribution",
)
display_aggregate(destination_counts_df)

## 5. Raw discharge-destination burden

The main table reports volume, LOS percentiles, final-unit dwell, threshold exceedance, excess bed-hours, burden shares, and the **72-hour burden ratio**:

\[
	ext{burden ratio}_{72} =
rac{	ext{share of all excess hours above 72h}}
{	ext{share of admissions}}
\]

- Above `1`: more long-stay burden than expected from volume
- Equal to `1`: burden proportional to volume
- Below `1`: less burden than expected from volume

In [ ]:
raw_destination_sql = make_base_ctes() + f"""
, destination_agg AS (
    SELECT
        discharge_location,
        COUNT(*) AS n_admissions,
        COUNTIF(final_unit_hours IS NOT NULL) AS n_with_final_unit,

        APPROX_QUANTILES(hospital_los_hours, 100)[OFFSET(50)]
            AS median_hospital_los_hours,
        APPROX_QUANTILES(hospital_los_hours, 100)[OFFSET(90)]
            AS p90_hospital_los_hours,
        APPROX_QUANTILES(hospital_los_hours, 100)[OFFSET(95)]
            AS p95_hospital_los_hours,

        APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(50)]
            AS median_final_unit_hours,
        APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(90)]
            AS p90_final_unit_hours,

        100 * AVG(CAST(hospital_los_hours > 48 AS INT64))
            AS pct_hospital_over_48h,
        100 * AVG(CAST(hospital_los_hours > 72 AS INT64))
            AS pct_hospital_over_72h,
        100 * AVG(CAST(hospital_los_hours > 168 AS INT64))
            AS pct_hospital_over_168h,
        100 * AVG(CAST(hospital_los_hours > 336 AS INT64))
            AS pct_hospital_over_336h,

        SUM(GREATEST(hospital_los_hours - 48, 0))
            AS excess_hospital_hours_48,
        SUM(GREATEST(hospital_los_hours - 72, 0))
            AS excess_hospital_hours_72,
        SUM(GREATEST(hospital_los_hours - 168, 0))
            AS excess_hospital_hours_168,
        SUM(GREATEST(hospital_los_hours - 336, 0))
            AS excess_hospital_hours_336,

        SUM(GREATEST(final_unit_hours - 48, 0))
            AS excess_final_unit_hours_48,
        SUM(GREATEST(final_unit_hours - 72, 0))
            AS excess_final_unit_hours_72
    FROM analysis_base
    GROUP BY discharge_location
    HAVING COUNT(*) = 0 OR COUNT(*) >= {MIN_CELL_SIZE}
),

totals AS (
    SELECT
        SUM(n_admissions) AS total_admissions,
        SUM(IF(discharge_location != 'UNKNOWN', n_admissions, 0))
            AS total_known_admissions,
        SUM(excess_hospital_hours_72)
            AS total_excess_hospital_hours_72,
        SUM(excess_final_unit_hours_72)
            AS total_excess_final_unit_hours_72
    FROM destination_agg
)

SELECT
    d.discharge_location,
    d.n_admissions,
    d.n_with_final_unit,

    ROUND(100 * SAFE_DIVIDE(
        d.n_admissions,
        t.total_admissions
    ), 2) AS pct_admissions_all,

    CASE
        WHEN d.discharge_location = 'UNKNOWN' THEN NULL
        ELSE ROUND(100 * SAFE_DIVIDE(
            d.n_admissions,
            t.total_known_admissions
        ), 2)
    END AS pct_admissions_known,

    ROUND(d.median_hospital_los_hours, 2)
        AS median_hospital_los_hours,
    ROUND(d.p90_hospital_los_hours, 2)
        AS p90_hospital_los_hours,
    ROUND(d.p95_hospital_los_hours, 2)
        AS p95_hospital_los_hours,
    ROUND(d.median_final_unit_hours, 2)
        AS median_final_unit_hours,
    ROUND(d.p90_final_unit_hours, 2)
        AS p90_final_unit_hours,

    ROUND(d.pct_hospital_over_48h, 2) AS pct_hospital_over_48h,
    ROUND(d.pct_hospital_over_72h, 2) AS pct_hospital_over_72h,
    ROUND(d.pct_hospital_over_168h, 2) AS pct_hospital_over_168h,
    ROUND(d.pct_hospital_over_336h, 2) AS pct_hospital_over_336h,

    ROUND(d.excess_hospital_hours_48, 0)
        AS excess_hospital_hours_48,
    ROUND(d.excess_hospital_hours_72, 0)
        AS excess_hospital_hours_72,
    ROUND(d.excess_hospital_hours_168, 0)
        AS excess_hospital_hours_168,
    ROUND(d.excess_hospital_hours_336, 0)
        AS excess_hospital_hours_336,

    ROUND(d.excess_final_unit_hours_48, 0)
        AS excess_final_unit_hours_48,
    ROUND(d.excess_final_unit_hours_72, 0)
        AS excess_final_unit_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        d.excess_hospital_hours_72,
        t.total_excess_hospital_hours_72
    ), 2) AS pct_total_excess_hospital_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        d.excess_final_unit_hours_72,
        t.total_excess_final_unit_hours_72
    ), 2) AS pct_total_excess_final_unit_hours_72,

    ROUND(
        SAFE_DIVIDE(
            SAFE_DIVIDE(
                d.excess_hospital_hours_72,
                t.total_excess_hospital_hours_72
            ),
            SAFE_DIVIDE(
                d.n_admissions,
                t.total_admissions
            )
        ),
        2
    ) AS burden_ratio_72

FROM destination_agg AS d
CROSS JOIN totals AS t
ORDER BY d.excess_hospital_hours_72 DESC
"""

raw_destination_df = run_query(
    raw_destination_sql,
    "Raw destination burden",
)
display_aggregate(
    raw_destination_df,
    count_columns=("n_admissions", "n_with_final_unit"),
)

In [ ]:
# Admission share versus share of excess hospital hours above 72h
plot_df = raw_destination_df[
    raw_destination_df["discharge_location"] != "UNKNOWN"
].copy().sort_values(
    "pct_total_excess_hospital_hours_72",
    ascending=True,
)

y = np.arange(len(plot_df))
height = 0.38

fig, ax = plt.subplots(figsize=(11, max(5, 0.55 * len(plot_df))))
ax.barh(
    y - height / 2,
    plot_df["pct_admissions_all"],
    height=height,
    label="Share of admissions",
)
ax.barh(
    y + height / 2,
    plot_df["pct_total_excess_hospital_hours_72"],
    height=height,
    label="Share of excess hospital hours >72h",
)
ax.set_yticks(y)
ax.set_yticklabels(plot_df["discharge_location"])
ax.set_xlabel("Percent")
ax.set_title("Admission Share vs Long-Stay Burden by Discharge Destination")
ax.legend()
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

# Burden-ratio chart
burden_plot_df = raw_destination_df[
    ~raw_destination_df["discharge_location"].isin(
        ["UNKNOWN", "DIED", "AGAINST ADVICE"]
    )
].copy()

horizontal_bar(
    burden_plot_df,
    category="discharge_location",
    value="burden_ratio_72",
    title="Long-Stay Burden Ratio Above 72 Hours",
    xlabel="Excess-hour share divided by admission share",
)

# Median final-unit dwell
final_unit_plot_df = raw_destination_df[
    raw_destination_df["n_with_final_unit"] >= MIN_CELL_SIZE
].copy()

horizontal_bar(
    final_unit_plot_df,
    category="discharge_location",
    value="median_final_unit_hours",
    title="Median Final-Care-Unit Dwell by Discharge Destination",
    xlabel="Median hours in final physical care unit",
)

## 6. Direct home-versus-coordinated-next-care comparison

`COORDINATED_NEXT_CARE` includes:

- home health care
- skilled nursing
- rehabilitation
- chronic/long-term acute care
- psychiatric facilities
- hospice
- assisted living
- another acute hospital
- other healthcare facilities

This grouped result is exploratory and must be interpreted alongside the granular destination table.

In [ ]:
comparison_sql = make_base_ctes() + f"""
, comparison_agg AS (
    SELECT
        comparison_group,
        COUNT(*) AS n_admissions,
        COUNTIF(final_unit_hours IS NOT NULL) AS n_with_final_unit,

        APPROX_QUANTILES(hospital_los_hours, 100)[OFFSET(50)]
            AS median_hospital_los_hours,
        APPROX_QUANTILES(hospital_los_hours, 100)[OFFSET(90)]
            AS p90_hospital_los_hours,
        APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(50)]
            AS median_final_unit_hours,
        APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(90)]
            AS p90_final_unit_hours,

        100 * AVG(CAST(hospital_los_hours > 72 AS INT64))
            AS pct_hospital_over_72h,
        100 * AVG(CAST(hospital_los_hours > 168 AS INT64))
            AS pct_hospital_over_168h,

        SUM(GREATEST(hospital_los_hours - 72, 0))
            AS excess_hospital_hours_72,
        SUM(GREATEST(final_unit_hours - 72, 0))
            AS excess_final_unit_hours_72
    FROM analysis_base
    GROUP BY comparison_group
    HAVING COUNT(*) = 0 OR COUNT(*) >= {MIN_CELL_SIZE}
),

totals AS (
    SELECT
        SUM(n_admissions) AS total_admissions,
        SUM(excess_hospital_hours_72)
            AS total_excess_hospital_hours_72,
        SUM(excess_final_unit_hours_72)
            AS total_excess_final_unit_hours_72
    FROM comparison_agg
)

SELECT
    c.comparison_group,
    c.n_admissions,
    c.n_with_final_unit,

    ROUND(100 * SAFE_DIVIDE(
        c.n_admissions,
        t.total_admissions
    ), 2) AS pct_admissions,

    ROUND(c.median_hospital_los_hours, 2)
        AS median_hospital_los_hours,
    ROUND(c.p90_hospital_los_hours, 2)
        AS p90_hospital_los_hours,
    ROUND(c.median_final_unit_hours, 2)
        AS median_final_unit_hours,
    ROUND(c.p90_final_unit_hours, 2)
        AS p90_final_unit_hours,

    ROUND(c.pct_hospital_over_72h, 2)
        AS pct_hospital_over_72h,
    ROUND(c.pct_hospital_over_168h, 2)
        AS pct_hospital_over_168h,

    ROUND(c.excess_hospital_hours_72, 0)
        AS excess_hospital_hours_72,
    ROUND(c.excess_final_unit_hours_72, 0)
        AS excess_final_unit_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        c.excess_hospital_hours_72,
        t.total_excess_hospital_hours_72
    ), 2) AS pct_total_excess_hospital_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        c.excess_final_unit_hours_72,
        t.total_excess_final_unit_hours_72
    ), 2) AS pct_total_excess_final_unit_hours_72,

    ROUND(
        SAFE_DIVIDE(
            SAFE_DIVIDE(
                c.excess_hospital_hours_72,
                t.total_excess_hospital_hours_72
            ),
            SAFE_DIVIDE(
                c.n_admissions,
                t.total_admissions
            )
        ),
        2
    ) AS burden_ratio_72

FROM comparison_agg AS c
CROSS JOIN totals AS t
ORDER BY c.excess_hospital_hours_72 DESC
"""

comparison_df = run_query(
    comparison_sql,
    "Home versus coordinated next care",
)
display_aggregate(
    comparison_df,
    count_columns=("n_admissions", "n_with_final_unit"),
)

comparison_plot_df = comparison_df[
    comparison_df["comparison_group"].isin(
        ["HOME_BASELINE", "COORDINATED_NEXT_CARE"]
    )
].copy()

for metric, title in [
    ("median_hospital_los_hours", "Median hospital LOS (hours)"),
    ("median_final_unit_hours", "Median final-unit dwell (hours)"),
    ("pct_hospital_over_72h", "Admissions over 72 hours (%)"),
    ("burden_ratio_72", "Burden ratio above 72 hours"),
]:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(
        comparison_plot_df["comparison_group"],
        comparison_plot_df[metric],
    )
    ax.set_title(title)
    ax.set_ylabel(title)
    ax.grid(axis="y", alpha=0.25)
    plt.xticks(rotation=15, ha="right")
    plt.tight_layout()
    plt.show()

## 7. Where does the burden accumulate?

This crosses the final physical care unit with the pathway group. It identifies units that may be acting as holding points for particular downstream pathways.

In [ ]:
unit_pathway_sql = make_base_ctes() + f"""
, unit_pathway_agg AS (
    SELECT
        COALESCE(final_careunit, 'UNKNOWN_FINAL_UNIT') AS final_careunit,
        pathway_group,
        COUNT(*) AS n_admissions,

        APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(50)]
            AS median_final_unit_hours,
        APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(90)]
            AS p90_final_unit_hours,

        100 * AVG(CAST(final_unit_hours > 72 AS INT64))
            AS pct_final_unit_over_72h,

        SUM(GREATEST(final_unit_hours - 72, 0))
            AS excess_final_unit_hours_72
    FROM analysis_base
    WHERE final_unit_hours IS NOT NULL
    GROUP BY final_careunit, pathway_group
    HAVING COUNT(*) = 0 OR COUNT(*) >= {MIN_CELL_SIZE}
),

unit_totals AS (
    SELECT
        final_careunit,
        SUM(excess_final_unit_hours_72)
            AS unit_total_excess_final_unit_hours_72
    FROM unit_pathway_agg
    GROUP BY final_careunit
)

SELECT
    u.final_careunit,
    u.pathway_group,
    u.n_admissions,
    ROUND(u.median_final_unit_hours, 2)
        AS median_final_unit_hours,
    ROUND(u.p90_final_unit_hours, 2)
        AS p90_final_unit_hours,
    ROUND(u.pct_final_unit_over_72h, 2)
        AS pct_final_unit_over_72h,
    ROUND(u.excess_final_unit_hours_72, 0)
        AS excess_final_unit_hours_72,
    ROUND(100 * SAFE_DIVIDE(
        u.excess_final_unit_hours_72,
        t.unit_total_excess_final_unit_hours_72
    ), 2) AS pct_of_unit_excess_final_hours_72

FROM unit_pathway_agg AS u
INNER JOIN unit_totals AS t
    USING (final_careunit)
ORDER BY u.excess_final_unit_hours_72 DESC
"""

unit_pathway_df = run_query(
    unit_pathway_sql,
    "Final care unit by discharge pathway",
)
display_aggregate(unit_pathway_df, rows=40)

top_unit_pathway = unit_pathway_df.copy()
top_unit_pathway["unit_pathway"] = (
    top_unit_pathway["final_careunit"].astype(str)
    + " — "
    + top_unit_pathway["pathway_group"].astype(str)
)

horizontal_bar(
    top_unit_pathway,
    category="unit_pathway",
    value="excess_final_unit_hours_72",
    title="Top Final-Unit / Discharge-Pathway Combinations",
    xlabel="Excess final-unit hours above 72h",
    top_n=25,
)

## 8. Within-service and ICU-status comparison

A hospital-wide destination comparison mixes discharge-process effects with clinical complexity.

This section partially reduces that problem by comparing ordinary home discharge with coordinated next care within the same:

- final clinical service
- ICU-exposure status

This remains descriptive rather than causal.

In [ ]:
service_comparison_sql = make_base_ctes() + f"""
SELECT
    COALESCE(final_service, 'UNKNOWN_SERVICE') AS final_service,
    comparison_group,
    had_icu,
    COUNT(*) AS n_admissions,

    APPROX_QUANTILES(hospital_los_hours, 100)[OFFSET(50)]
        AS median_hospital_los_hours,
    APPROX_QUANTILES(final_unit_hours, 100)[OFFSET(50)]
        AS median_final_unit_hours,

    100 * AVG(CAST(hospital_los_hours > 72 AS INT64))
        AS pct_hospital_over_72h,

    SUM(GREATEST(hospital_los_hours - 72, 0))
        AS excess_hospital_hours_72,
    SUM(GREATEST(final_unit_hours - 72, 0))
        AS excess_final_unit_hours_72

FROM analysis_base
WHERE comparison_group IN (
    'HOME_BASELINE',
    'COORDINATED_NEXT_CARE'
)
GROUP BY final_service, comparison_group, had_icu
HAVING COUNT(*) = 0 OR COUNT(*) >= {MIN_CELL_SIZE}
ORDER BY excess_final_unit_hours_72 DESC
"""

service_comparison_df = run_query(
    service_comparison_sql,
    "Within-service and ICU-status comparison",
)

for column in [
    "median_hospital_los_hours",
    "median_final_unit_hours",
    "pct_hospital_over_72h",
    "excess_hospital_hours_72",
    "excess_final_unit_hours_72",
]:
    service_comparison_df[column] = service_comparison_df[column].round(2)

display_aggregate(service_comparison_df, rows=50)

# Aggregate the already-aggregated ICU strata to create a compact service view.
service_summary = (
    service_comparison_df
    .groupby(["final_service", "comparison_group"], as_index=False)
    .agg(
        n_admissions=("n_admissions", "sum"),
        median_hospital_los_hours=("median_hospital_los_hours", "median"),
        median_final_unit_hours=("median_final_unit_hours", "median"),
        excess_final_unit_hours_72=("excess_final_unit_hours_72", "sum"),
    )
)

home_service = (
    service_summary[
        service_summary["comparison_group"] == "HOME_BASELINE"
    ]
    .drop(columns="comparison_group")
    .rename(columns={
        "n_admissions": "n_home",
        "median_hospital_los_hours": "home_median_hospital_los_hours",
        "median_final_unit_hours": "home_median_final_unit_hours",
        "excess_final_unit_hours_72": "home_excess_final_unit_hours_72",
    })
)

next_service = (
    service_summary[
        service_summary["comparison_group"] == "COORDINATED_NEXT_CARE"
    ]
    .drop(columns="comparison_group")
    .rename(columns={
        "n_admissions": "n_next_care",
        "median_hospital_los_hours": "next_care_median_hospital_los_hours",
        "median_final_unit_hours": "next_care_median_final_unit_hours",
        "excess_final_unit_hours_72": "next_care_excess_final_unit_hours_72",
    })
)

service_gap_df = home_service.merge(next_service, on="final_service", how="inner")

service_gap_df["hospital_median_gap_hours"] = (
    service_gap_df["next_care_median_hospital_los_hours"]
    - service_gap_df["home_median_hospital_los_hours"]
)

service_gap_df["final_unit_median_gap_hours"] = (
    service_gap_df["next_care_median_final_unit_hours"]
    - service_gap_df["home_median_final_unit_hours"]
)

service_gap_df = service_gap_df.sort_values(
    "final_unit_median_gap_hours",
    ascending=False,
)

assert_no_small_cells(
    service_gap_df,
    count_columns=("n_home", "n_next_care"),
)
display(service_gap_df.head(30))

service_gap_plot = service_gap_df[
    (service_gap_df["n_home"] >= 100)
    & (service_gap_df["n_next_care"] >= 100)
].copy()

horizontal_bar(
    service_gap_plot,
    category="final_service",
    value="final_unit_median_gap_hours",
    title=(
        "Within-Service Difference in Median Final-Unit Dwell\n"
        "Coordinated Next Care Minus Home"
    ),
    xlabel="Difference in median final-unit hours",
    top_n=20,
)

## 9. Sensitivity to extreme durations

Excess-hour totals can be dominated by a tiny number of extreme records.

This repeats the 72-hour burden calculation after winsorizing hospital LOS and final-unit dwell at the global P99.9. A conclusion is more robust when the same destinations remain overrepresented.

In [ ]:
sensitivity_sql = make_base_ctes() + f"""
, bounds AS (
    SELECT
        APPROX_QUANTILES(hospital_los_hours, 1000)[OFFSET(999)]
            AS hospital_p999,
        APPROX_QUANTILES(final_unit_hours, 1000)[OFFSET(999)]
            AS final_unit_p999
    FROM analysis_base
),

destination_sensitivity AS (
    SELECT
        a.discharge_location,
        COUNT(*) AS n_admissions,

        SUM(GREATEST(a.hospital_los_hours - 72, 0))
            AS raw_excess_hospital_hours_72,

        SUM(GREATEST(
            LEAST(a.hospital_los_hours, b.hospital_p999) - 72,
            0
        )) AS winsorized_excess_hospital_hours_72,

        SUM(GREATEST(a.final_unit_hours - 72, 0))
            AS raw_excess_final_unit_hours_72,

        SUM(GREATEST(
            LEAST(a.final_unit_hours, b.final_unit_p999) - 72,
            0
        )) AS winsorized_excess_final_unit_hours_72

    FROM analysis_base AS a
    CROSS JOIN bounds AS b
    GROUP BY a.discharge_location
    HAVING COUNT(*) = 0 OR COUNT(*) >= {MIN_CELL_SIZE}
),

totals AS (
    SELECT
        SUM(n_admissions) AS total_admissions,
        SUM(raw_excess_hospital_hours_72)
            AS total_raw_excess_hospital_hours_72,
        SUM(winsorized_excess_hospital_hours_72)
            AS total_winsorized_excess_hospital_hours_72,
        SUM(raw_excess_final_unit_hours_72)
            AS total_raw_excess_final_unit_hours_72,
        SUM(winsorized_excess_final_unit_hours_72)
            AS total_winsorized_excess_final_unit_hours_72
    FROM destination_sensitivity
)

SELECT
    d.discharge_location,
    d.n_admissions,

    ROUND(100 * SAFE_DIVIDE(
        d.raw_excess_hospital_hours_72,
        t.total_raw_excess_hospital_hours_72
    ), 2) AS raw_pct_excess_hospital_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        d.winsorized_excess_hospital_hours_72,
        t.total_winsorized_excess_hospital_hours_72
    ), 2) AS winsorized_pct_excess_hospital_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        d.raw_excess_final_unit_hours_72,
        t.total_raw_excess_final_unit_hours_72
    ), 2) AS raw_pct_excess_final_unit_hours_72,

    ROUND(100 * SAFE_DIVIDE(
        d.winsorized_excess_final_unit_hours_72,
        t.total_winsorized_excess_final_unit_hours_72
    ), 2) AS winsorized_pct_excess_final_unit_hours_72,

    ROUND(
        SAFE_DIVIDE(
            SAFE_DIVIDE(
                d.winsorized_excess_hospital_hours_72,
                t.total_winsorized_excess_hospital_hours_72
            ),
            SAFE_DIVIDE(
                d.n_admissions,
                t.total_admissions
            )
        ),
        2
    ) AS winsorized_burden_ratio_72

FROM destination_sensitivity AS d
CROSS JOIN totals AS t
ORDER BY winsorized_pct_excess_hospital_hours_72 DESC
"""

sensitivity_df = run_query(
    sensitivity_sql,
    "P99.9 sensitivity analysis",
)
display_aggregate(sensitivity_df)

## 10. Automated evidence summary

In [ ]:
def get_group_row(
    df: pd.DataFrame,
    column: str,
    value: str,
) -> pd.Series | None:
    match = df[df[column] == value]
    return None if match.empty else match.iloc[0]


home = get_group_row(
    comparison_df,
    "comparison_group",
    "HOME_BASELINE",
)
next_care = get_group_row(
    comparison_df,
    "comparison_group",
    "COORDINATED_NEXT_CARE",
)
unknown = get_group_row(
    comparison_df,
    "comparison_group",
    "UNKNOWN",
)

print("=" * 78)
print("EVIDENCE SUMMARY")
print("=" * 78)

if home is not None and next_care is not None:
    print(
        f"Coordinated next care represents "
        f"{next_care['pct_admissions']:.2f}% of admissions and "
        f"{next_care['pct_total_excess_hospital_hours_72']:.2f}% of "
        f"hospital excess hours above 72h."
    )
    print(
        f"Its 72h burden ratio is {next_care['burden_ratio_72']:.2f}; "
        f"the home baseline ratio is {home['burden_ratio_72']:.2f}."
    )
    print(
        f"Median hospital LOS: "
        f"{next_care['median_hospital_los_hours']:.2f}h for coordinated "
        f"next care versus {home['median_hospital_los_hours']:.2f}h for home."
    )
    print(
        f"Median final-unit dwell: "
        f"{next_care['median_final_unit_hours']:.2f}h for coordinated "
        f"next care versus {home['median_final_unit_hours']:.2f}h for home."
    )

if unknown is not None:
    print(
        f"Unknown discharge destination represents "
        f"{unknown['pct_admissions']:.2f}% of admissions."
    )

if not service_gap_df.empty:
    repeated_positive = (
        service_gap_df["final_unit_median_gap_hours"] > 0
    ).mean() * 100
    print(
        f"Across services with both comparison groups, coordinated next-care "
        f"discharges had a higher median final-unit dwell in "
        f"{repeated_positive:.1f}% of services."
    )

print()
print("Interpretation:")
print(
    "A burden ratio above 1, longer final-unit dwell than home, and a repeated "
    "within-service pattern provide BIDMC-specific evidence consistent with "
    "a downstream-care transition bottleneck."
)
print(
    "The analysis cannot identify the exact hours spent medically ready and "
    "waiting. That requires operational readiness, referral, authorization, "
    "acceptance, and placement timestamps."
)

## 11. Decision framework

### Strong evidence consistent with a next-care bottleneck

The hypothesis gains substantial BIDMC-specific support when:

1. Coordinated next-care discharges account for a larger share of excess bed-hours than of admissions.
2. Their burden ratio is meaningfully above 1.
3. Their final-unit dwell is longer than ordinary home discharges.
4. The pattern persists across multiple services and ICU/non-ICU strata.
5. Results remain similar after P99.9 sensitivity analysis.
6. The pattern is driven by skilled nursing, rehabilitation, long-term acute care, psychiatric care, or home health—not only death or transfer to another acute hospital.

### Mixed evidence

The result is mixed when total hospital LOS is longer for next-care groups but final-unit dwell is similar to home, or the effect occurs only in a few specialized services. Clinical complexity may explain much of the difference.

### Defensible final wording if results are supportive

> Admissions requiring coordinated downstream care account for a disproportionate share of prolonged hospital occupancy and final-unit dwell at BIDMC. The pattern persists across multiple services and is consistent with constraints in transitions to post-acute and community care. Because MIMIC-IV does not record medical-readiness or placement-process timestamps, the analysis does not estimate the portion of time directly caused by waiting.

### Recommended portfolio outputs

1. Admission share versus excess-hour share by destination
2. Burden ratio above 72 hours
3. Median final-unit dwell by destination
4. Top final-unit/pathway combinations
5. Within-service final-unit dwell difference: next care minus home
6. A concise conclusion with the causal limitation stated explicitly